# scikit-verify: see the math your code computes

`to_sympy(fn, *args)` runs your NumPy function once and returns the **symbolic formula** it computed — the actual mathematics, extracted from the actual execution.

No annotations, no DSL, no rewriting your code.

In [1]:
import numpy as np
import sympy
from skverify import to_sympy

## 1. A first trace

What does `scipy.integrate.trapezoid` actually compute?

In [2]:
from scipy.integrate import trapezoid

y = np.linspace(0, 1, 8) ** 2
t = to_sympy(lambda y: trapezoid(y, dx=0.1), y)
t.formula

0.05*y[0] + 0.05*y[7] + Sum(0.1*y[j + 1], (j, 0, 5))

The half-weight endpoints sit **outside** the `Sum` — the formula shows the trapezoid rule's structure, not just its number. And the number is still there:

In [3]:
t.value, trapezoid(y, dx=0.1)

(np.float64(0.2357142857142857), np.float64(0.2357142857142857))

## 2. Your own code — and a proof it's right

A heat-equation step, written by hand. Is the update rule really the discrete Laplacian?

In [4]:
def step(u, dt, h):
    lap = (u[2:] - 2 * u[1:-1] + u[:-2]) / h**2
    unew = u.copy()
    unew[1:-1] = u[1:-1] + dt * lap
    return unew

s = to_sympy(step, np.sin(np.linspace(0, np.pi, 9)), 0.01, 0.1)
s.formula

Piecewise((dt*(u[i + 1] + u[i - 1] - 2*u[i])/h**2 + u[i], (i >= 1) & (i < 8)), (u[i], True))

Two things happened:

- `dt` and `h` stayed **symbolic** — floats lift as parameters, named after your arguments.
- The boundary handling is *visible*: interior points get the stencil, endpoints stay untouched.

Now prove it equals the rule from the textbook:

In [5]:
from skverify.checks import against
from skverify.helpers import axis_idx

i = axis_idx(0)
U = sympy.IndexedBase("u")
dt, h = sympy.symbols("dt h", real=True)

textbook = sympy.Piecewise(
    (U[i] + dt * (U[i + 1] - 2 * U[i] + U[i - 1]) / h**2, (i >= 1) & (i < 8)),
    (U[i], True),
)
against(s, textbook)

Evidence(verdict='proven', method='canonical', detail=0)

**`proven`** — symbolic equality, not a numerical spot-check.

## 3. Free gradients

Trace a hand-rolled cross-entropy loss, then differentiate the *formula* with plain sympy:

In [6]:
def loss(w, x, t):
    p = 1.0 / (1.0 + np.exp(-(x @ w)))
    return -np.sum(t * np.log(p) + (1 - t) * np.log(1 - p))

x = np.array([[1.0, 2.0], [1.0, -1.0], [1.0, 0.5]])
t = np.array([1.0, 0.0, 1.0])
L = to_sympy(loss, np.array([0.1, -0.2]), x, t)
L.formula

-Sum((1 - t[j])*log(1 - 1.0/(1.0 + exp(-Sum(w[k]*x[j, k], (k, 0, 1))))) + log(1.0/(1.0 + exp(-Sum(w[k]*x[j, k], (k, 0, 1)))))*t[j], (j, 0, 2))

In [7]:
W = sympy.IndexedBase("w")
grad0 = sympy.simplify(sympy.diff(L.formula.doit(), W[0]))

# check against finite differences
subs = {sympy.IndexedBase("x")[idx]: float(v) for idx, v in np.ndenumerate(x)}
subs |= {sympy.IndexedBase("t")[k]: float(v) for k, v in enumerate(t)}
g = sympy.lambdify((W[0], W[1]), grad0.xreplace(subs))

eps = 1e-6
fd = (loss([0.1 + eps, -0.2], x, t) - loss([0.1 - eps, -0.2], x, t)) / (2 * eps)
g(0.1, -0.2), fd

(np.float64(-0.5000000000000001), np.float64(-0.500000000069889))

## 4. Find a bug by *subtracting formulas*

I implemented Simpson's rule from a paper — and typo'd a weight (`3` where it should be `4`). Values only say "wrong". Formulas say **where**:

In [8]:
from scipy.integrate import simpson

def my_simpson(y, h):
    total = y[0] + y[-1]
    for k in range(1, len(y) - 1):
        total = total + (3 if k % 2 == 1 else 2) * y[k]   # bug: 3 should be 4
    return total * h / 3

y9 = np.linspace(0, 1, 9) ** 2
mine = to_sympy(my_simpson, y9, 0.125)
ref = to_sympy(lambda y: simpson(y, dx=0.125), y9)

diff = (mine.formula - ref.formula).doit().subs(sympy.Symbol("h", real=True), sympy.Rational(1, 8))
sympy.nsimplify(sympy.expand(diff), rational=True)

-y[1]/24 - y[3]/24 - y[5]/24 - y[7]/24

Every surviving term is an **odd index** with coefficient $-1/24 = \frac{3-4}{3}h$: the bug is in the odd-$k$ branch, and the weight is off by exactly one.

## 5. Honesty is the contract

Certificates state their own scope. Data-dependent branches become **preconditions**; results the trace couldn't derive are **disclosed**, never hidden:

In [9]:
def model(u):
    if u.max() > 2.0:
        return u * 0.5
    return u * 2.0

r = to_sympy(model, np.array([1.0, 3.0]))
r.formula, r.preconditions

(0.5*u[i], Max(u[0], u[1]) > 2.0)

In [10]:
from scipy.interpolate import make_interp_spline

def fit(x, y):
    return make_interp_spline(x, y)

x8 = np.linspace(0, 7, 8)
spl = to_sympy(fit, x8, np.sin(x8))
spl.c[2].formula

dgbsv_1_2[2, 0]

In [11]:
# the banded solve is a COMPILED routine: it becomes a named atom,
# and its residual A @ x == b was checked on this very call
[(entry[0], dict(entry[1])) for entry in spl.unchecked]

[('coloc', {'contract': 'unknown'}), ('dgbsv', {'residual': 'ok'})]

`residual: ok` — the LAPACK result was **verified against its defining equation** during the trace, not trusted.

---

### Where to go next

- `result.derivation()` — the full computation as readable steps
- `result.preconditions` / `result.unchecked` — what the certificate assumes
- `skverify.checks` — `against`, `banded`, `conserves_mass`, ...

Works today on real code from **scipy** (splines, quadrature, stats), **statsmodels** (OLS/WLS, autocovariance), and **sklearn** (metrics, kernels, SVM decision functions).